# Face Mesh and Facial Landmark Analysis

**Objective:** To detect facial landmarks using MediaPipe and analyze facial features in real time.

In [2]:
!pip install --no-cache-dir mediapipe


In [4]:
import cv2
import mediapipe as mp
import math

print("Libraries imported successfully!")
print("MediaPipe version:", mp.__version__)

Libraries imported successfully!
MediaPipe version: 1.0.1


In [6]:
import mediapipe as mp

print("MediaPipe version:", mp.__version__)
print("MediaPipe location:", mp.__file__)

MediaPipe version: 1.0.1
MediaPipe location: C:\Users\user121\anaconda3\New folder\Lib\site-packages\mediapipe\__init__.py


In [7]:
print(dir(mp))

['Image', 'ImageFormat', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'tasks']


In [8]:
!pip install mediapipe

In [9]:
import mediapipe as mp

print("MediaPipe Tasks available:", hasattr(mp, "tasks"))
print("Tasks:", dir(mp.tasks))

MediaPipe Tasks available: True
Tasks: ['BaseOptions', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'audio', 'components', 'text', 'vision']


In [10]:
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

print("Face Landmarker API loaded successfully!")

Face Landmarker API loaded successfully!


In [11]:
import os

model_path = "face_landmarker.task"

print("Model path:", os.path.abspath(model_path))
print("Model exists:", os.path.exists(model_path))

Model path: C:\Users\user121\face_landmarker.task
Model exists: False


In [12]:
import urllib.request

model_url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task"

urllib.request.urlretrieve(model_url, "face_landmarker.task")

print("Face Landmarker model downloaded successfully!")

Face Landmarker model downloaded successfully!


In [13]:
import os

print("Model exists:", os.path.exists("face_landmarker.task"))

if os.path.exists("face_landmarker.task"):
    print("Model size:", round(os.path.getsize("face_landmarker.task") / (1024 * 1024), 2), "MB")

Model exists: True
Model size: 3.58 MB


In [14]:
import mediapipe as mp

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="face_landmarker.task"
    ),
    running_mode=VisionRunningMode.IMAGE,
    num_faces=2
)

face_landmarker = FaceLandmarker.create_from_options(options)

print("Face Landmarker initialized successfully!")

Face Landmarker initialized successfully!


In [15]:
import cv2
import mediapipe as mp

cap = cv2.VideoCapture(0)

print("Face Landmarker started.")
print("Press 'q' to quit.")

while True:
    success, frame = cap.read()

    if not success:
        print("Could not access camera.")
        break

    frame = cv2.flip(frame, 1)

    # Convert OpenCV frame to MediaPipe Image
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    # Detect faces and landmarks
    result = face_landmarker.detect(mp_image)

    # Draw detected landmarks
    if result.face_landmarks:
        for landmarks in result.face_landmarks:

            h, w, _ = frame.shape

            for landmark in landmarks:
                x = int(landmark.x * w)
                y = int(landmark.y * h)

                cv2.circle(
                    frame,
                    (x, y),
                    1,
                    (0, 255, 0),
                    -1
                )

    cv2.imshow("MediaPipe Face Landmarker", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

print("Face Landmarker completed.")

Face Landmarker started.
Press 'q' to quit.
Face Landmarker completed.


In [16]:
import cv2
import mediapipe as mp
import math

cap = cv2.VideoCapture(0)

print("Eye Distance Analysis started.")
print("Press 'q' to quit.")

while True:
    success, frame = cap.read()

    if not success:
        print("Could not access camera.")
        break

    # Mirror the webcam
    frame = cv2.flip(frame, 1)

    # Convert BGR to RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Create MediaPipe Image
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    # Detect facial landmarks
    result = face_landmarker.detect(mp_image)

    if result.face_landmarks:

        for landmarks in result.face_landmarks:

            h, w, _ = frame.shape

            # Select two eye landmarks
            left_eye = landmarks[33]
            right_eye = landmarks[263]

            # Convert normalized coordinates to pixels
            left_point = (
                int(left_eye.x * w),
                int(left_eye.y * h)
            )

            right_point = (
                int(right_eye.x * w),
                int(right_eye.y * h)
            )

            # Calculate distance between the two points
            eye_distance = math.sqrt(
                (right_point[0] - left_point[0]) ** 2 +
                (right_point[1] - left_point[1]) ** 2
            )

            # Draw eye points
            cv2.circle(
                frame,
                left_point,
                6,
                (255, 0, 0),
                -1
            )

            cv2.circle(
                frame,
                right_point,
                6,
                (255, 0, 0),
                -1
            )

            # Draw line between the eyes
            cv2.line(
                frame,
                left_point,
                right_point,
                (0, 255, 255),
                2
            )

            # Display eye distance
            cv2.putText(
                frame,
                f"Eye Distance: {eye_distance:.2f} pixels",
                (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2
            )

            # Display number of landmarks
            cv2.putText(
                frame,
                f"Landmarks: {len(landmarks)}",
                (20, 75),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 255, 255),
                2
            )

            # Draw additional facial landmarks
            for landmark in landmarks:
                x = int(landmark.x * w)
                y = int(landmark.y * h)

                cv2.circle(
                    frame,
                    (x, y),
                    1,
                    (0, 255, 0),
                    -1
                )

    cv2.imshow("Face Landmark and Eye Distance Analysis", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

print("Eye Distance Analysis completed.")

Eye Distance Analysis started.
Press 'q' to quit.
Eye Distance Analysis completed.


## Conclusion

In this practical, MediaPipe Face Landmarker was used to detect facial
landmarks in real time. Selected eye landmarks were used to calculate
the distance between the eyes in pixels. The facial landmarks and
calculated distance were displayed on the live webcam feed, showing
how computer vision can be used for facial feature analysis.